# Telangana Tourism — Complete Data Analysis
This notebook analyzes the two cleaned CSV files supplied for the Telangana tourism project.
It covers descriptive statistics, visitor totals, domestic vs foreign analysis, monthly trends, destination rankings, category analysis, revenue analysis, visualizations, and key insights.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

monthly = pd.read_csv("Monthly_Visitor_Trends.csv")
places = pd.read_csv("Web_App_Place_Comparisons.csv")

print("Monthly shape:", monthly.shape)
print("Places shape:", places.shape)
display(monthly.head())
display(places.head())


In [ ]:
print("Monthly columns:")
print(monthly.columns.tolist())
print("\nPlace columns:")
print(places.columns.tolist())

print("\nMissing values — Monthly")
display(monthly.isna().sum())

print("\nMissing values — Places")
display(places.isna().sum())


In [ ]:
print("Monthly descriptive statistics")
display(monthly.describe())

print("Place descriptive statistics")
display(places.describe())


In [ ]:
total_visitors = monthly["Total Visitors"].sum()
domestic = monthly["Domestic Visitors"].sum()
foreign = monthly["Foreign Visitors"].sum()
revenue = monthly["Est Revenue (Lakhs INR)"].sum()

print(f"Total visitors: {total_visitors:,.0f}")
print(f"Domestic visitors: {domestic:,.0f}")
print(f"Foreign visitors: {foreign:,.0f}")
print(f"Estimated revenue: {revenue:,.2f} Lakhs INR")
print(f"Domestic share: {domestic/total_visitors*100:.2f}%")
print(f"Foreign share: {foreign/total_visitors*100:.2f}%")


In [ ]:
# If the 24 rows are Jan-Dec 2024 followed by Jan-Dec 2025
if len(monthly) >= 24 and "Year" not in monthly.columns:
    monthly["Year"] = ["2024"]*12 + ["2025"]*(len(monthly)-12)

if "Year" in monthly.columns:
    yearly = monthly.groupby("Year")[["Total Visitors","Domestic Visitors","Foreign Visitors","Est Revenue (Lakhs INR)"]].sum()
    display(yearly)
    if "2024" in yearly.index and "2025" in yearly.index:
        growth = (yearly.loc["2025","Total Visitors"]-yearly.loc["2024","Total Visitors"])/yearly.loc["2024","Total Visitors"]*100
        print(f"2024→2025 visitor growth: {growth:.2f}%")


In [ ]:
highest = monthly.loc[monthly["Total Visitors"].idxmax()]
lowest = monthly.loc[monthly["Total Visitors"].idxmin()]
print("Highest month:")
display(highest.to_frame().T)
print("Lowest month:")
display(lowest.to_frame().T)


In [ ]:
top10 = places.sort_values("Total Footfall", ascending=False).head(10)
display(top10[["Rank","District / Place","Category","Total Footfall","Est Revenue (Lakhs INR)"]])


In [ ]:
category_summary = places.groupby("Category", as_index=False).agg(
    Total_Footfall=("Total Footfall","sum"),
    Estimated_Revenue_Lakhs=("Est Revenue (Lakhs INR)","sum"),
    Places=("District / Place","count")
).sort_values("Total_Footfall", ascending=False)
display(category_summary)


In [ ]:
plt.figure(figsize=(11,6))
if "Year" in monthly.columns:
    for year, g in monthly.groupby("Year", sort=False):
        plt.plot(g["Month"], g["Total Visitors"], marker="o", label=str(year))
    plt.legend()
else:
    plt.plot(monthly["Month"], monthly["Total Visitors"], marker="o")
plt.title("Monthly Total Visitors in Telangana")
plt.xlabel("Month"); plt.ylabel("Visitors"); plt.grid(alpha=.25)
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
plt.bar(["Domestic","Foreign"], [domestic, foreign])
plt.title("Domestic vs Foreign Visitors")
plt.ylabel("Visitors")
plt.show()


In [ ]:
top10_plot = places.sort_values("Total Footfall", ascending=False).head(10).sort_values("Total Footfall")
plt.figure(figsize=(10,6))
plt.barh(top10_plot["District / Place"], top10_plot["Total Footfall"])
plt.title("Top 10 Listed Destinations by Footfall")
plt.xlabel("Total Footfall")
plt.show()


In [ ]:
rev10 = places.sort_values("Est Revenue (Lakhs INR)", ascending=False).head(10).sort_values("Est Revenue (Lakhs INR)")
plt.figure(figsize=(10,6))
plt.barh(rev10["District / Place"], rev10["Est Revenue (Lakhs INR)"])
plt.title("Top 10 Listed Destinations by Estimated Revenue")
plt.xlabel("Revenue (Lakhs INR)")
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
plt.bar(category_summary["Category"], category_summary["Total_Footfall"])
plt.title("Footfall by Tourism Category")
plt.xlabel("Category"); plt.ylabel("Total Footfall")
plt.xticks(rotation=35, ha="right")
plt.show()


In [ ]:
print("KEY FINDINGS")
print("1. Highest monthly visitor count:", highest["Month"], int(highest["Total Visitors"]))
print("2. Lowest monthly visitor count:", lowest["Month"], int(lowest["Total Visitors"]))
top = places.sort_values("Total Footfall", ascending=False).iloc[0]
print("3. Top listed destination:", top["District / Place"], int(top["Total Footfall"]))
print("4. Domestic visitor share:", round(domestic/total_visitors*100,2), "%")
print("5. Foreign visitor share:", round(foreign/total_visitors*100,2), "%")
